In [27]:
import numpy as np
import pandas as pd
import math
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_log_error
from statsmodels.tsa.deterministic import DeterministicProcess
from sklearn.model_selection import train_test_split

In [2]:
# path to the dataset in Kaggle's notebook
# Change to the path that stores your files
path = '../data/'

### 1. Compute Moving Average of Oil Prices

In [3]:
# read oil price
data_oil = pd.read_csv(path + 'oil.csv', parse_dates=['date'], infer_datetime_format=True, index_col='date')

########################################################################################################################
# TODO: compute data_oil['ma_oil'] as the moving average of data_oil['dcoilwtico'] with window size 7
# Hint: check the documentation of .rolling() method of pandas.DataFrame
########################################################################################################################
data_oil['ma_oil'] = data_oil['dcoilwtico'].rolling(7).mean()


# Create continguous moving average of oil prices
calendar = pd.DataFrame(index=pd.date_range('2013-01-01', '2017-08-31'))

########################################################################################################################
# TODO 1: merge two DataFrame instances (data_oil and calendar) such that the merged instances has the same indexes
# as calendar.
# TODO 2: replace each NaN in data_oil['ma_oil'] by the first non-null value before it.
# Hint: check the documentation of .merge() and .fillna() methods of pandas.DataFrame
########################################################################################################################
calendar = calendar.merge(data_oil, how='outer', left_index=True, right_index=True)
calendar['ma_oil'] = calendar['ma_oil'].fillna(method="ffill")

calendar.head(15)

C:\anaconda\lib\site-packages\pandas\io\parsers\base_parser.py:1070: UserWarning: Parsing '14/10/2013' in DD/MM/YYYY format. Provide format or specify infer_datetime_format=True for consistent parsing.
  return tools.to_datetime(
C:\anaconda\lib\site-packages\pandas\io\parsers\base_parser.py:1070: UserWarning: Parsing '15/10/2013' in DD/MM/YYYY format. Provide format or specify infer_datetime_format=True for consistent parsing.
  return tools.to_datetime(
C:\anaconda\lib\site-packages\pandas\io\parsers\base_parser.py:1070: UserWarning: Parsing '16/10/2013' in DD/MM/YYYY format. Provide format or specify infer_datetime_format=True for consistent parsing.
  return tools.to_datetime(
C:\anaconda\lib\site-packages\pandas\io\parsers\base_parser.py:1070: UserWarning: Parsing '17/10/2013' in DD/MM/YYYY format. Provide format or specify infer_datetime_format=True for consistent parsing.
  return tools.to_datetime(
C:\anaconda\lib\site-packages\pandas\io\parsers\base_parser.py:1070: UserWarning

,dcoilwtico,ma_oil
2013-01-01,NaN,NaN
2013-01-02,97.46,96.737143
2013-01-03,90.71,92.408571
2013-01-04,97.10,92.408571
2013-01-05,90.74,92.032857
2013-01-06,NaN,92.032857
2013-01-07,97.94,95.842857
2013-01-08,107.93,105.202857
2013-01-09,NaN,105.202857
2013-01-10,102.09,102.847143


### 2. Create Workday Feature

In [4]:
########################################################################################################################
# TODO: create a True/False feature calendar['wd'] to indicate whether each date is a workday (Monday-Friday) or not.
# Hint: check documentation of pandas.DatetimeIndex.dayofweek
########################################################################################################################
calendar['wd'] = calendar.index.dayofweek < 5
calendar.head(15) # display some entries of calendar

,dcoilwtico,ma_oil,wd
2013-01-01,NaN,NaN,True
2013-01-02,97.46,96.737143,True
2013-01-03,90.71,92.408571,True
2013-01-04,97.10,92.408571,True
2013-01-05,90.74,92.032857,False
2013-01-06,NaN,92.032857,False
2013-01-07,97.94,95.842857,True
2013-01-08,107.93,105.202857,True
2013-01-09,NaN,105.202857,True
2013-01-10,102.09,102.847143,True


### 3. Read Train and Test Data

In [5]:
df_train = pd.read_csv(path + 'train.csv',
                       usecols=['store_nbr', 'family', 'date', 'sales'],
                       dtype={'store_nbr': 'category', 'family': 'category', 'sales': 'float32'},
                       parse_dates=['date'], infer_datetime_format=True)

df_train.date = df_train.date.dt.to_period('D')
df_train = df_train.set_index(['store_nbr', 'family', 'date']).sort_index()

df_train.head(15) # display some entries of the training data

sales
store_nbr family     date             
1         AUTOMOTIVE 2013-01-01    0.0
                     2013-01-02    2.0
                     2013-01-03    3.0
                     2013-01-04    3.0
                     2013-01-05    5.0
                     2013-01-06    2.0
                     2013-01-07    0.0
                     2013-01-08    2.0
                     2013-01-09    2.0
                     2013-01-10    2.0
                     2013-01-11    3.0
                     2013-01-12    2.0
                     2013-01-13    2.0
                     2013-01-14    2.0
                     2013-01-15    1.0

In [6]:
df_test = pd.read_csv(path + 'test.csv',
                      usecols=['store_nbr', 'family', 'date'],
                      dtype={'store_nbr': 'category', 'family': 'category'},
                      parse_dates=['date'], infer_datetime_format=True)

df_test.date = df_test.date.dt.to_period('D')
df_test = df_test.set_index(['store_nbr', 'family', 'date']).sort_index()

df_test.head(15) # display some entries of the testing data

Empty DataFrame
Columns: []
Index: [(1, AUTOMOTIVE, 2017-08-16), (1, AUTOMOTIVE, 2017-08-17), (1, AUTOMOTIVE, 2017-08-18), (1, AUTOMOTIVE, 2017-08-19), (1, AUTOMOTIVE, 2017-08-20), (1, AUTOMOTIVE, 2017-08-21), (1, AUTOMOTIVE, 2017-08-22), (1, AUTOMOTIVE, 2017-08-23), (1, AUTOMOTIVE, 2017-08-24), (1, AUTOMOTIVE, 2017-08-25), (1, AUTOMOTIVE, 2017-08-26), (1, AUTOMOTIVE, 2017-08-27), (1, AUTOMOTIVE, 2017-08-28), (1, AUTOMOTIVE, 2017-08-29), (1, AUTOMOTIVE, 2017-08-30)]

In [7]:
# set the range of data used in training
sdate = '2017-04-01'
edate = '2017-08-15'

# we will train a model that takes feature of a date as input and predicts the sales for each store and family of goods on that date.
y = df_train.unstack(['store_nbr', 'family']).loc[sdate:edate]


########################################################################################################################
# TODO: create the trend feature X: the value for sdate is 1, the value for the next day of sdate is 2, etc.
# Hint: check the documentation of DeterministicProcess, or this tutorial: https://www.kaggle.com/code/ryanholbrook/trend.
########################################################################################################################
dp = DeterministicProcess(index=y.index, constant=True, order=1, drop=True) # change 'None' to your answer
X = dp.in_sample()

# Extentions
X['oil']  = calendar.loc[sdate:edate]['ma_oil'].values
X['wd']   = calendar.loc[sdate:edate]['wd'].values

X.head(15)

,const,trend,oil,wd
date,,,,
2017-04-01,1.0,1.0,48.570000,False
2017-04-02,1.0,2.0,48.570000,False
2017-04-03,1.0,3.0,48.570000,True
2017-04-04,1.0,4.0,49.561429,True
2017-04-05,1.0,5.0,48.187143,True
2017-04-06,1.0,6.0,48.187143,True
2017-04-07,1.0,7.0,48.187143,True
2017-04-08,1.0,8.0,49.481429,False
2017-04-09,1.0,9.0,49.481429,False


In [46]:
#Split data into training and validation sets
#Note: Time series data might not be suitable for us to slice training and validation datasets randomly
#X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.1, random_state=42)

totalRows, totalColumns = X.shape
train_percentage=0.9

#Split X, first 90% dedicated for training, last 10% for val
X_train=X[:math.floor(totalRows*train_percentage)]
X_val=X[math.floor(totalRows*train_percentage):]

#Split y, first 90% dedicated for training, last 10% for val
y_train=y[:math.floor(totalRows*train_percentage)]
y_val=y[math.floor(totalRows*train_percentage):]

### 4. Train Model!

In [43]:
model = LinearRegression()
model.fit(X_train, y_train)
y_pred = pd.DataFrame(model.predict(X_val), index=X_val.index, columns=y.columns)

In [48]:
# Results on the training set

y_pred   = y_pred.stack(['store_nbr', 'family']).reset_index()
y_target = y_val.stack(['store_nbr', 'family']).reset_index().copy()

y_target['sales_pred'] = y_pred['sales'].clip(0.) # Sales should be >= 0

########################################################################################################################
# TODO: show the training loss for each type of product.
# Hint: check the documentation of DataFrame.groupby() and GroupBy.apply().
########################################################################################################################
y_target.groupby('family').apply(lambda r: mean_squared_log_error(r['sales'], r['sales_pred'])).sort_values(ascending=False)

family
SCHOOL AND OFFICE SUPPLIES    1.865053
LIQUOR,WINE,BEER              0.446249
LINGERIE                      0.429689
GROCERY II                    0.370310
CELEBRATION                   0.343829
HARDWARE                      0.318061
AUTOMOTIVE                    0.296298
BEAUTY                        0.290124
SEAFOOD                       0.288545
LADIESWEAR                    0.288090
LAWN AND GARDEN               0.278119
PLAYERS AND ELECTRONICS       0.266560
MAGAZINES                     0.266274
HOME AND KITCHEN I            0.247748
HOME AND KITCHEN II           0.239810
PET SUPPLIES                  0.219885
EGGS                          0.182173
PRODUCE                       0.167799
HOME APPLIANCES               0.159054
FROZEN FOODS                  0.146363
CLEANING                      0.146081
BEVERAGES                     0.136316
GROCERY I                     0.132631
MEATS                         0.132521
PREPARED FOODS                0.130017
POULTRY           

In [44]:
# Test predictions

stest = '2017-08-16'
etest = '2017-08-31'

########################################################################################################################
# TODO: create the feature matrix of test data.
# Hint: check the documentation of DeterministicProcess.
########################################################################################################################
X_test = dp.out_of_sample(steps = 16) # from 8.16 - 8.31
X_test['oil']  = calendar.loc[stest:etest]['ma_oil'].values 
X_test['wd']   = calendar.loc[stest:etest]['wd'].values
print(X_test)

sales_pred = pd.DataFrame(model.predict(X_test), index=X_test.index, columns=y.columns)
sales_pred = sales_pred.stack(['store_nbr', 'family'])

sales_pred[sales_pred < 0] = 0. # Sales should be >= 0

            const  trend        oil     wd
2017-08-16    1.0  138.0  48.281429   True
2017-08-17    1.0  139.0  47.995714   True
2017-08-18    1.0  140.0  47.852857   True
2017-08-19    1.0  141.0  47.852857  False
2017-08-20    1.0  142.0  47.852857  False
2017-08-21    1.0  143.0  47.688571   True
2017-08-22    1.0  144.0  47.522857   True
2017-08-23    1.0  145.0  47.645714   True
2017-08-24    1.0  146.0  47.598571   True
2017-08-25    1.0  147.0  47.720000   True
2017-08-26    1.0  148.0  47.720000  False
2017-08-27    1.0  149.0  47.720000  False
2017-08-28    1.0  150.0  47.624286   True
2017-08-29    1.0  151.0  47.320000   True
2017-08-30    1.0  152.0  47.115714   True
2017-08-31    1.0  153.0  47.060000   True


In [45]:
# Create submission

df_sub = pd.read_csv(path + 'sample_submission.csv', index_col='id')
df_sub.sales = sales_pred.values
df_sub.to_csv('submission.csv', index=True)